# 02 — NLP Pipeline Experiments (Phase 2)

Verifies the Phase 2 processing modules (`cleaner`, `nlp_pipeline`, `categoriser`, `scorer`) built per `CLAUDE.md` Section 6.2 and Section 13's Phase 2 build order.

**Data available at time of writing:** only Google Trends (372 rows, 5 tracked keywords, one keyword with zero volume). Reddit ingestion is blocked by new-account restrictions; the News ingester hasn't been built yet (a key is configured in `.env`, but the ingester itself is deferred until after this pipeline works, per project decision). So `raw_content` is empty — no real document text exists yet.

Given that, this notebook does two things:
1. Runs `cleaner` / `nlp_pipeline` / `categoriser` on **representative mock text** (per `CLAUDE.md` instruction #8: build with mock data when blocked on a source, and continue) so the text pipeline is proven correct before real documents arrive.
2. Runs `scorer`'s momentum formula on the **real** Google Trends data already in the database — that part needs no text at all.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.processing import cleaner, nlp_pipeline, categoriser, scorer
from src.storage import db

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Cleaner

In [2]:
raw = "<p>Loving the QUIET LUXURY aesthetic rn https://vogue.com/article café society vibes!! #streetwear</p>"
print(cleaner.clean_text(raw))

loving the quiet luxury aesthetic rn cafe society vibes streetwear


## 2. NLP pipeline (mock documents)

Three representative posts, styled like what the Reddit/News ingesters will eventually pull in from r/femalefashionadvice and fashion press.

In [3]:
MOCK_DOCUMENTS = [
    "Loving the quiet luxury aesthetic this season \u2014 oversized blazers, cargo pants, "
    "and chunky loafers are everywhere on r/femalefashionadvice right now.",
    "Barbiecore is officially dead, dark academia and mob wife aesthetic are the new streetwear "
    "staples for fall according to street style photographers at fashion week.",
    "Les blazers oversize sont partout cet automne, avec des pantalons larges et des mocassins.",
]

for doc in MOCK_DOCUMENTS:
    result = nlp_pipeline.process_document(doc)
    print(f"language={result['language']}")
    print(f"cleaned: {result['cleaned_text'][:80]}...")
    print(f"keywords: {[k['keyword'] for k in result['keywords']]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 34483.82it/s]

2026-09-09 17:28:52.702 | INFO     | src.processing.nlp_pipeline:process_document:62 - Skipping keyword extraction for non-English document


language=en
cleaned: loving the quiet luxury aesthetic this season oversized blazers cargo pants and ...
keywords: ['oversized blazers', 'loafers femalefashionadvice', 'cargo pants', 'luxury aesthetic', 'season oversized', 'quiet luxury', 'pants chunky', 'aesthetic season', 'blazers cargo', 'chunky loafers']

language=en
cleaned: barbiecore is officially dead dark academia and mob wife aesthetic are the new s...
keywords: ['barbiecore', 'barbiecore officially', 'academia mob', 'mob wife', 'dark academia', 'mob', 'dead dark', 'fashion', 'wife aesthetic', 'fashion week']

language=other
cleaned: les blazers oversize sont partout cet automne avec des pantalons larges et des m...
keywords: []



**Observation:** the third document (French) is correctly detected as non-English and skipped for keyword extraction — v1 is English-only per `CLAUDE.md`'s known limitations.

## 3. Categoriser: mapping extracted keywords to the taxonomy

In [4]:
sample_doc = MOCK_DOCUMENTS[0]
result = nlp_pipeline.process_document(sample_doc)

for kw in result["keywords"]:
    match = categoriser.categorise_keyword(kw["keyword"])
    print(f"{kw['keyword']:<30} -> {match['category']} ({match['match_type']})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11787.86it/s]

2026-09-09 17:28:58.703 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'oversized blazers' did not match taxonomy — flagged as emerging


2026-09-09 17:28:58.711 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'loafers femalefashionadvice' did not match taxonomy — flagged as emerging


2026-09-09 17:28:58.719 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'luxury aesthetic' did not match taxonomy — flagged as emerging


...

2026-09-09 17:28:58.738 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'pants chunky' did not match taxonomy — flagged as emerging


2026-09-09 17:28:58.744 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'aesthetic season' did not match taxonomy — flagged as emerging


2026-09-09 17:28:58.750 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'blazers cargo' did not match taxonomy — flagged as emerging


2026-09-09 17:28:58.760 | INFO     | src.processing.categoriser:categorise_keyword:154 - Keyword 'chunky loafers' did not match taxonomy — flagged as emerging


oversized blazers              -> None (emerging)
loafers femalefashionadvice    -> None (emerging)
cargo pants                    -> Clothing Items (exact)
luxury aesthetic               -> None (emerging)
season oversized               -> None (emerging)
quiet luxury                   -> Aesthetics (exact)
pants chunky                   -> None (emerging)
aesthetic season               -> None (emerging)
blazers cargo                  -> None (emerging)
chunky loafers                 -> None (emerging)


**Observation (known limitation, not a bug):** KeyBERT extracts multi-word phrases ("oversized blazers", "chunky loafers"), but the taxonomy in `CLAUDE.md` Section 5 lists single terms ("oversized", "loafers"). Whole-phrase exact/fuzzy matching doesn't catch a taxonomy term embedded inside a longer extracted phrase, so most compound phrases fall through to `emerging` even when a human would obviously categorise them. Single-word or already-canonical phrases ("cargo pants", "quiet luxury") match cleanly via exact match.

This is worth revisiting once real Reddit/News text is flowing — either by extracting unigrams alongside bigrams, or by checking whether any taxonomy term is a substring of the extracted phrase before falling back to fuzzy/semantic matching.

## 4. Scorer: momentum scoring on real Google Trends data

In [5]:
engine = db.init_db()
categoriser.seed_categories(engine)

signals = scorer.run()
for s in signals:
    print(s)

2026-09-09 17:28:58.767 | INFO     | src.storage.db:init_db:31 - Database initialised at sqlite:///data/fashion_trends.db


2026-09-09 17:28:58.768 | INFO     | src.processing.categoriser:seed_categories:72 - Seeded 6 taxonomy categories


2026-09-09 17:28:58.769 | INFO     | src.storage.db:init_db:31 - Database initialised at sqlite:///data/fashion_trends.db


2026-09-09 17:28:58.770 | INFO     | src.processing.scorer:score_keyword_from_google_trends:138 - Scored 'oversized': momentum=0.0434 status=stable


2026-09-09 17:28:58.771 | INFO     | src.processing.scorer:score_keyword_from_google_trends:138 - Scored 'cargo pants': momentum=0.0387 status=stable


2026-09-09 17:28:58.772 | INFO     | src.processing.scorer:score_keyword_from_google_trends:138 - Scored 'Y2K': momentum=-0.02 status=stable


2026-09-09 17:28:58.773 | INFO     | src.processing.scorer:score_keyword_from_google_trends:138 - Scored 'quiet luxury': momentum=2.5224 status=emerging


2026-09-09 17:28:58.773 | INFO     | src.processing.scorer:run:164 - Trend scoring complete: 4 signals computed


{'keyword': 'oversized', 'category_id': 2, 'date': '2026-08-31', 'mention_count': 65, 'momentum_score': 0.0434, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'cargo pants', 'category_id': 1, 'date': '2026-08-31', 'mention_count': 69, 'momentum_score': 0.0387, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'Y2K', 'category_id': 5, 'date': '2026-08-31', 'mention_count': 70, 'momentum_score': -0.02, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'quiet luxury', 'category_id': 5, 'date': '2026-08-31', 'mention_count': 34, 'momentum_score': 2.5224, 'source_diversity': 1, 'trend_status': 'emerging'}


**Observation:** Google Trends' 0-100 interest scale sits well below the `mention_count` bands `classify_trend()` uses (100-1000 for "rising", >1000 for "peak"), which are tuned for raw document counts. So even a keyword with strong momentum (like `quiet luxury` at momentum > 2.5) can still land as `emerging` rather than `rising`/`peak` — these bands will start behaving as intended once Reddit/News mention counts (which run into the hundreds/thousands) are flowing into `trend_signals` alongside Google Trends.

Also note `source_diversity=1` for every signal right now — Google Trends is the only source ingested so far, so the diversity bonus in the momentum formula is inert until Reddit/News join in.

## 5. Summary: what's real vs. mock in this notebook

| Component | Status |
|---|---|
| `cleaner.clean_text` | Real logic, demoed on synthetic text |
| `nlp_pipeline` (language detect + keyword extraction) | Real logic, tested against mock documents (no real `raw_content` yet) |
| `categoriser` (exact/fuzzy/semantic taxonomy matching) | Real logic, tested against mock keywords |
| `scorer` (momentum + classification) | **Real data** — 372 real Google Trends rows, real trend_signals written to the DB |

**Next steps (per project decision):** build the NewsAPI ingester (key already in `.env`) and revisit Reddit once account restrictions clear, then re-run this notebook against real `raw_content` to validate the keyword-extraction/categorisation observations above with live data.